# buahaha ? Pelatihan prediksi penjualan 7 hari

Unggah notebook ini ke Google Colab, gunakan runtime **CPU**, lalu jalankan sel berurutan dari atas. Unggah CSV ketika diminta. Notebook tidak mengubah data aplikasi buahaha.

Target: **penjualan tercatat dalam kg**, bukan permintaan yang tidak terpenuhi. Data sintetis hanya membuktikan alur prototipe. Hari stok habis dapat membatasi penjualan; model ini belum mengoreksi permintaan yang hilang.

Hasil: evaluasi, prediksi harian, simulasi stok, model Python, dan metadata dalam satu ZIP. Belum ada integrasi otomatis ke dashboard atau interval prediksi terkalibrasi. Tanggal prediksi mengikuti tanggal terakhir CSV, bukan tanggal menjalankan notebook.

In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "joblib>=1.3,<2"

In [ ]:
import io, json, hashlib, platform, zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from threadpoolctl import threadpool_limits
from IPython.display import display

SEED = 42
HORIZON = 7
print('pandas:', pd.__version__, '| sklearn:', sklearn.__version__)

## 1. Unggah dan periksa CSV
Pilih `buahaha_penjualan_dummy_2021-10-13_2026-10-12.csv`. Tanggal harus lengkap setiap hari untuk setiap SKU. Hari tidak berjualan tetap memerlukan baris nol; hari yang hilang tidak otomatis dianggap nol.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
assert len(csv_names) == 1, 'Unggah tepat satu CSV.'
source_name = csv_names[0]
raw = uploaded[source_name]
df = pd.read_csv(io.BytesIO(raw), dtype={'sku': str})
required = ['tanggal', 'sku', 'nama_buah', 'satuan', 'jumlah_terjual_kg',
            'stok_awal_kg', 'stok_masuk_kg', 'stok_rusak_kg', 'stok_akhir_kg',
            'stok_habis', 'operasional', 'jenis_data']
assert set(required).issubset(df.columns), 'Kolom wajib belum lengkap.'
assert not df[required].isna().any().any(), 'Ada nilai kosong pada kolom wajib.'
df['tanggal'] = pd.to_datetime(df['tanggal'], format='%Y-%m-%d', errors='raise')
assert not df.duplicated(['tanggal', 'sku']).any(), 'Ada tanggal-SKU ganda.'
assert df['satuan'].eq('kg').all(), 'Notebook ini menggunakan kg.'
numeric = ['jumlah_terjual_kg', 'stok_awal_kg', 'stok_masuk_kg', 'stok_rusak_kg', 'stok_akhir_kg']
for col in numeric:
    df[col] = pd.to_numeric(df[col], errors='raise')
assert np.isfinite(df[numeric].to_numpy()).all() and df[numeric].ge(0).all().all()
assert df['operasional'].isin([0, 1]).all() and df['stok_habis'].isin([0, 1]).all()
df = df.sort_values(['sku', 'tanggal']).reset_index(drop=True)
calendar = pd.date_range(df['tanggal'].min(), df['tanggal'].max(), freq='D')
assert len(calendar) >= 180, 'Sediakan minimal 180 hari untuk pembagian contoh ini.'
for sku, g in df.groupby('sku'):
    assert list(g['tanggal']) == list(calendar), f'Tanggal SKU {sku} tidak lengkap/seragam.'
    assert g['nama_buah'].nunique() == 1, f'Nama berubah pada SKU {sku}.'
    assert np.allclose(g['stok_awal_kg'].iloc[1:], g['stok_akhir_kg'].iloc[:-1]), f'Stok terputus: {sku}'
assert np.allclose(df.stok_awal_kg + df.stok_masuk_kg - df.jumlah_terjual_kg - df.stok_rusak_kg, df.stok_akhir_kg), 'Saldo stok tidak sesuai.'
print(f'{len(df):,} baris | {df.sku.nunique()} SKU | {len(calendar)} hari')
print('Rentang:', calendar[0].date(), 'sampai', calendar[-1].date())
print('Baris stok habis:', int(df.stok_habis.sum()), '| jenis data:', df.jenis_data.unique())
display(df.head())

## 2. Bentuk fitur historis dan target
Satu model bersama untuk semua SKU, dengan horizon 1?7 sebagai fitur. Semua fitur berasal dari informasi yang sudah tersedia pada **akhir tanggal asal prediksi**. Penjualan hari itu sudah final; jika digunakan sebelum tutup hari, gunakan tanggal sebelumnya.

Target tidak dipakai sebagai fitur. Harga/diskon masa depan, stok akhir masa depan, dan omzet masa depan tidak dimasukkan. Baseline memakai hari yang sama minggu sebelumnya. Tidak ada pembagian data acak atau early stopping dengan validasi acak.

In [ ]:
sku_codes = {sku: i for i, sku in enumerate(sorted(df.sku.unique()))}
features = ['sku_code', 'horizon', 'hari_tujuan', 'bulan_tujuan',
            'jual_0', 'jual_1', 'jual_6', 'jual_7', 'jual_13', 'jual_27',
            'rata_7', 'rata_28', 'std_7', 'habis_7', 'buka_7']

def make_samples(data):
    parts = []
    for sku, g in data.groupby('sku', sort=True):
        g = g.sort_values('tanggal').reset_index(drop=True)
        q = g['jumlah_terjual_kg']
        base = pd.DataFrame({'sku': sku, 'sku_code': sku_codes[sku], 'origin': g.tanggal})
        for lag in [0, 1, 6, 7, 13, 27]:
            base[f'jual_{lag}'] = q.shift(lag)
        base['rata_7'] = q.rolling(7).mean()
        base['rata_28'] = q.rolling(28).mean()
        base['std_7'] = q.rolling(7).std()
        base['habis_7'] = g.stok_habis.rolling(7).sum()
        base['buka_7'] = g.operasional.rolling(7).sum()
        for h in range(1, HORIZON + 1):
            p = base.copy()
            p['horizon'] = h
            p['target_date'] = p.origin + pd.Timedelta(days=h)
            p['hari_tujuan'] = p.target_date.dt.dayofweek
            p['bulan_tujuan'] = p.target_date.dt.month
            p['target'] = q.shift(-h)
            p['baseline'] = q.shift(7 - h)
            parts.append(p)
    return pd.concat(parts, ignore_index=True).dropna(subset=features + ['baseline'])

samples = make_samples(df)
last_date = df.tanggal.max()
validation_cut = last_date - pd.Timedelta(days=56)
test_cut = last_date - pd.Timedelta(days=28)
print('Validasi:', (validation_cut + pd.Timedelta(days=1)).date(), 'sampai', test_cut.date())
print('Uji akhir:', (test_cut + pd.Timedelta(days=1)).date(), 'sampai', last_date.date())
print('Setiap periode memakai 4 prediksi mingguan, masing-masing 7 hari tanpa target tumpang tindih.')

## 3. Latih, pilih metode dengan validasi, lalu uji sekali pada periode terakhir
MAE = rata-rata selisih prediksi dalam kg (lebih kecil lebih baik). WAPE = total selisih absolut dibagi total penjualan aktual; ini **bukan persentase akurasi**. WAPE tidak tersedia jika total aktual nol.

Pemilihan metode memakai validasi 28 hari. Periode uji 28 hari setelahnya tidak dipakai untuk memilih metode. Pada evaluasi mingguan, fitur boleh memakai penjualan yang telah terjadi sampai tanggal asal prediksi, tetapi model tetap dibekukan sepanjang periode evaluasi.

In [ ]:
def fit_until(cut):
    train = samples[(samples.target_date <= cut) & samples.target.notna()]
    assert len(train) > 0 and train.target_date.max() <= cut
    model = HistGradientBoostingRegressor(
        learning_rate=0.07, max_iter=120, max_leaf_nodes=15,
        l2_regularization=1.0, categorical_features=['sku_code'],
        early_stopping=False, random_state=SEED,
    )
    with threadpool_limits(limits=2):
        model.fit(train[features], train.target)
    return model

def predict(model, frame):
    with threadpool_limits(limits=2):
        return np.maximum(0, model.predict(frame[features]))

def evaluate_period(model, cut):
    origins = [cut + pd.Timedelta(days=7 * week) for week in range(4)]
    evaluation = samples[samples.origin.isin(origins) & samples.target.notna()].copy()
    assert len(evaluation) == len(sku_codes) * 28
    evaluation['ml'] = predict(model, evaluation)
    return evaluation

def metrics(frame, pred):
    actual = frame.target.to_numpy()
    predicted = frame[pred].to_numpy()
    denominator = float(np.abs(actual).sum())
    return {'MAE_kg': float(mean_absolute_error(actual, predicted)),
            'WAPE_persen': float(np.abs(actual - predicted).sum() / denominator * 100) if denominator else None}

validation_model = fit_until(validation_cut)
validation = evaluate_period(validation_model, validation_cut)
validation_scores = pd.DataFrame({name: metrics(validation, name) for name in ['baseline', 'ml']}).T
chosen = min(['baseline', 'ml'], key=lambda name: metrics(validation, name)['MAE_kg'])
print('Evaluasi VALIDASI:')
display(validation_scores)
print('Metode dipilih sebelum melihat uji akhir:', chosen)

test_model = fit_until(test_cut)
test = evaluate_period(test_model, test_cut)
test['selected'] = test[chosen]
print('Evaluasi UJI AKHIR (jangan dipakai untuk memilih ulang model):')
display(pd.DataFrame({name: metrics(test, name) for name in ['baseline', 'ml', 'selected']}).T)
per_sku = pd.DataFrame([{'sku': sku, **metrics(g, 'selected')} for sku, g in test.groupby('sku')])
display(per_sku)

## 4. Latih ulang dengan semua riwayat dan prediksi 7 hari ke depan
Model akhir boleh belajar sampai tanggal terakhir CSV karena evaluasi sudah selesai. Metode terpilih tidak diganti setelah melihat uji akhir. Jika baseline menang, hasil produksi contoh menggunakan baseline; model ML tetap disertakan untuk penelitian lanjutan.

In [ ]:
final_model = fit_until(last_date)
future = samples[samples.origin.eq(last_date)].copy()
assert len(future) == len(sku_codes) * HORIZON
future['ml'] = predict(final_model, future)
future['prediksi_kg'] = future[chosen]
forecast = future[['sku', 'target_date', 'prediksi_kg']].rename(columns={'target_date': 'tanggal'})
forecast['nama_buah'] = forecast.sku.map(df.groupby('sku').nama_buah.first())
forecast = forecast.sort_values(['sku', 'tanggal']).reset_index(drop=True)
forecast['metode'] = chosen
assert np.isfinite(forecast.prediksi_kg).all() and forecast.prediksi_kg.ge(0).all()
display(forecast.head(14))
ax = forecast.pivot(index='tanggal', columns='nama_buah', values='prediksi_kg').plot(figsize=(12, 5), marker='o')
ax.set(title='buahaha ? perkiraan penjualan 7 hari (simulasi)', xlabel='Tanggal', ylabel='kg per hari')
ax.figure.tight_layout()
plt.show()

## 5. Simulasi stok
Gunakan saldo akhir CSV sebagai stok awal simulasi. Asumsi: tidak ada pembelian tambahan, pesanan terikat, retur, kerusakan baru, atau jadwal tutup khusus selama horizon. Ini perkiraan **penjualan tercatat**, belum estimasi permintaan tanpa batas stok.

Untuk operasional, ganti dengan stok aktual yang tanggalnya cocok, tambahkan rencana stok masuk, alokasikan pesanan yang sudah dipesan tanpa menghitungnya dua kali, dan perhitungkan kerusakan. Jangan mengartikan stok cukup dalam 7 hari sebagai tanggal habis pasti setelah hari ke-7.

In [ ]:
stocks = df.sort_values('tanggal').groupby('sku').tail(1).set_index('sku')['stok_akhir_kg']
stock_rows = []
for sku, group in forecast.groupby('sku'):
    initial = float(stocks.loc[sku])
    daily = group.sort_values('tanggal')
    cumulative = daily.prediksi_kg.cumsum()
    exhausted = daily.loc[cumulative >= initial, 'tanggal']
    shortage = daily.loc[cumulative > initial, 'tanggal']
    required = float(daily.prediksi_kg.sum())
    stock_rows.append({
        'sku': sku, 'nama_buah': daily.nama_buah.iloc[0],
        'stok_awal_simulasi_kg': initial,
        'kebutuhan_7_hari_kg': required,
        'sisa_setelah_7_hari_kg': max(0.0, initial - required),
        'kekurangan_7_hari_kg': max(0.0, required - initial),
        'tanggal_diperkirakan_habis': last_date.strftime('%Y-%m-%d') if initial == 0 else (exhausted.iloc[0].strftime('%Y-%m-%d') if len(exhausted) else None),
        'tanggal_mulai_kekurangan': shortage.iloc[0].strftime('%Y-%m-%d') if len(shortage) else None,
        'status': 'Stok sudah kosong' if initial == 0 else ('Diperkirakan kurang dalam 7 hari' if required > initial else 'Diperkirakan cukup untuk 7 hari'),
    })
stock_forecast = pd.DataFrame(stock_rows)
display(stock_forecast.round(2))

## 6. Simpan dan unduh hasil
Unduh ZIP sebelum sesi Colab berakhir. JSON/CSV berisi hasil untuk integrasi dashboard; `.joblib` adalah model Python dan tidak bisa langsung dijalankan oleh React/browser. API atau proses Python diperlukan untuk menjalankan model pada data baru. Baseline yang terpilih memerlukan histori 7 hari, sedangkan fitur ML memerlukan minimal 28 hari lengkap per SKU.

Notebook ini belum membuat API atau menu Prediksi di buahaha. Jangan menerapkan hasil dummy sebagai rekomendasi pembelian nyata. File model hanya boleh dimuat dari sumber tepercaya.

In [ ]:
output = Path('buahaha_hasil_prediksi')
output.mkdir(exist_ok=True)
export_forecast = forecast.copy()
export_forecast['tanggal'] = export_forecast.tanggal.dt.strftime('%Y-%m-%d')
export_forecast.to_csv(output / 'prediksi_penjualan_7_hari.csv', index=False)
stock_forecast.to_csv(output / 'prediksi_stok.csv', index=False)
test.to_csv(output / 'hasil_uji_akhir.csv', index=False)
per_sku.to_csv(output / 'evaluasi_per_sku.csv', index=False)
metadata = {
    'schema_version': 1, 'project': 'buahaha',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_file': source_name, 'source_sha256': hashlib.sha256(raw).hexdigest(),
    'source_types': sorted(df.jenis_data.unique().tolist()),
    'trained_through': last_date.strftime('%Y-%m-%d'),
    'horizon_days': HORIZON, 'selected_method': chosen,
    'validation_metrics': {name: metrics(validation, name) for name in ['baseline', 'ml']},
    'test_metrics': {name: metrics(test, name) for name in ['baseline', 'ml', 'selected']},
    'versions': {'python': platform.python_version(), 'sklearn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__},
    'limitations': ['Data sintetis bukan validasi usaha nyata', 'Target adalah penjualan tercatat, bukan permintaan yang hilang',
                    'Simulasi stok tanpa pengadaan dan kerusakan baru', 'Belum memiliki interval prediksi terkalibrasi'],
}
results = {'metadata': metadata, 'sales': export_forecast.to_dict(orient='records'),
           'stock': stock_rows}
(output / 'prediksi_buahaha.json').write_text(json.dumps(results, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
joblib.dump({'model': final_model, 'selected_method': chosen, 'features': features,
             'sku_codes': sku_codes, 'metadata': metadata}, output / 'model_buahaha.joblib')
# Simpan histori minimum agar fitur dan baseline dapat direkonstruksi.
df.groupby('sku', group_keys=False).tail(28).to_csv(output / 'histori_28_hari.csv', index=False)
(output / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
(output / 'requirements-versi.txt').write_text(chr(10).join([
    f'pandas=={pd.__version__}', f'numpy=={np.__version__}',
    f'scikit-learn=={sklearn.__version__}', f'joblib=={joblib.__version__}'
]), encoding='utf-8')
zip_path = Path('buahaha_hasil_prediksi.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(output.iterdir()):
        if file.is_file():
            archive.write(file, arcname=file.name)
files.download(str(zip_path))
print('Selesai. Simpan ZIP dan notebook ke komputer/Drive Anda.')

## Rujukan
- [Google Colab: unggah dan unduh berkas](https://colab.research.google.com/notebooks/io.ipynb)
- [scikit-learn: fitur historis untuk peramalan](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html)
- [HistGradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)